In [ ]:
# !pip install --upgrade pip
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn

## Importações de Bibliotecas

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

## Renomear Colunas — `rename()`

In [ ]:
df = pd.DataFrame({
    'Nome Completo': ['Ana Silva', 'Bruno Costa', 'Carlos Lima', 'Diana Melo'],
    'Idade (anos)':  [25, 30, 35, 28],
    'Salario R$':    [3000, 4500, 5200, 3800],
    'CIDADE':        ['SP', 'RJ', 'MG', 'SP']
})

df

In [ ]:
df.columns.tolist()

#### Método 1 — `rename()`: renomear colunas específicas

In [ ]:
df1 = df.rename(columns={
    'Nome Completo': 'nome',
    'Salario R$':    'salario'
})

df1.columns.tolist()

#### Método 2 — substituir todas as colunas de uma vez

In [ ]:
df2 = df.copy()
df2.columns = ['nome', 'idade', 'salario', 'cidade']

df2.columns.tolist()

#### Método 3 — padronizar com função `str` (ideal após leitura de CSV)

In [ ]:
df3 = df.copy()

df3.columns = (
    df3.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

df3.columns.tolist()

In [ ]:
df3

## Remover Duplicatas — `drop_duplicates()`

In [ ]:
df = pd.DataFrame({
    'nome':   ['Ana', 'Bruno', 'Ana', 'Carlos', 'Bruno', 'Ana'],
    'cidade': ['SP',  'RJ',   'SP',  'MG',     'RJ',    'SP'],
    'compra': [150,    200,   150,    320,      200,     180]
})

df

#### Diagnosticar duplicatas antes de remover

In [ ]:
print(f'Total de linhas: {len(df)}')
print(f'Duplicatas encontradas: {df.duplicated().sum()}')

In [ ]:
# Visualizar apenas as linhas duplicadas
df[df.duplicated(keep=False)]

#### Remover duplicatas — todas as colunas

In [ ]:
df_sem_dup = df.drop_duplicates()

print(f'Linhas restantes: {len(df_sem_dup)}')
df_sem_dup

#### Remover duplicatas por colunas específicas — parâmetro `subset`

In [ ]:
# Considera duplicata quem tiver mesmo nome E cidade — mantém a primeira ocorrência
df.drop_duplicates(subset=['nome', 'cidade'], keep='first')

In [ ]:
# Mantendo a última ocorrência
df.drop_duplicates(subset=['nome', 'cidade'], keep='last')

In [ ]:
# keep=False — remove TODAS as ocorrências duplicadas
df.drop_duplicates(keep=False)

---
## Converter Tipos — `astype()`

In [ ]:
# Simulando leitura de CSV — colunas com tipos incorretos
df = pd.DataFrame({
    'nome':    ['Ana', 'Bruno', 'Carlos', 'Diana'],
    'idade':   ['25', '30', '35', '28'],           # deveria ser int
    'salario': [3000.0, 4500.0, 5200.0, 3800.0],   # deveria ser int
    'cidade':  ['SP', 'RJ', 'MG', 'SP']            # ideal: category
})

print(df.dtypes)

In [ ]:
print(f'Memória antes: {df.memory_usage(deep=True).sum()} bytes')

#### `object` → `int`

In [ ]:
df['idade'] = df['idade'].astype(int)

print(df['idade'].dtype)
df['idade']

#### `float` → `int`

In [ ]:
df['salario'] = df['salario'].astype(int)

print(df['salario'].dtype)
df['salario']

#### `object` → `category`

In [ ]:
df['cidade'] = df['cidade'].astype('category')

print(f'Categorias únicas: {df["cidade"].cat.categories.tolist()}')
print(f'Códigos internos:  {df["cidade"].cat.codes.tolist()}')

In [ ]:
print(df.dtypes)
print(f'\nMemória depois: {df.memory_usage(deep=True).sum()} bytes')

#### Tratando valores inválidos — `pd.to_numeric()` com `errors='coerce'`

In [ ]:
serie = pd.Series(['25', '30', 'N/A', '28', 'desconhecido'])

# errors='coerce' transforma inválidos em NaN em vez de quebrar
pd.to_numeric(serie, errors='coerce')

In [ ]:
# Int64 (I maiúsculo) — nullable integer: aceita NaN e mantém tipo inteiro
pd.to_numeric(serie, errors='coerce').astype('Int64')

In [ ]:
# Converter múltiplas colunas de uma vez
df2 = pd.DataFrame({
    'idade':   ['25', '30', '35'],
    'salario': [3000.0, 4500.0, 5200.0],
    'uf':      ['SP', 'RJ', 'MG']
})

df2 = df2.astype({'idade': int, 'salario': int, 'uf': 'category'})
print(df2.dtypes)

---
## Trabalhar com Datas — `pd.to_datetime()` e acessor `.dt`

In [ ]:
df = pd.DataFrame({
    'cliente':     ['Ana', 'Bruno', 'Carlos', 'Diana', 'Eduardo'],
    'data_compra': ['2024-01-15', '2024-03-22', '2024-07-08',
                    '2023-11-30', '2024-09-01'],
    'valor':       [350, 120, 890, 450, 230]
})

print(df.dtypes)
df

#### Converter string → datetime

In [ ]:
df['data_compra'] = pd.to_datetime(df['data_compra'])

print(df['data_compra'].dtype)

#### Formato brasileiro — `dd/mm/aaaa`

In [ ]:
datas_br = pd.Series(['15/01/2024', '22/03/2024', '08/07/2024'])
pd.to_datetime(datas_br, format='%d/%m/%Y')

#### Extrair componentes com o acessor `.dt`

In [ ]:
df['ano']        = df['data_compra'].dt.year
df['mes']        = df['data_compra'].dt.month
df['dia']        = df['data_compra'].dt.day
df['dia_semana'] = df['data_compra'].dt.day_name()
df['trimestre']  = df['data_compra'].dt.quarter
df['semana_iso'] = df['data_compra'].dt.isocalendar().week

df[['cliente', 'data_compra', 'ano', 'mes', 'dia', 'dia_semana', 'trimestre', 'semana_iso']]

#### Filtrar por período

In [ ]:
# Somente compras de 2024
df[df['data_compra'].dt.year == 2024][['cliente', 'data_compra', 'valor']]

In [ ]:
# Primeiro semestre de 2024
mask = (df['data_compra'] >= '2024-01-01') & (df['data_compra'] <= '2024-06-30')
df[mask][['cliente', 'data_compra', 'valor']]

#### Tratamento de datas inválidas — `errors='coerce'` e `NaT`

In [ ]:
serie = pd.Series(['2024-01-15', 'data inválida', '2024-07-08', 'N/A'])
convertida = pd.to_datetime(serie, errors='coerce')

print(convertida)
print(f'\nQuantidade de NaT: {convertida.isna().sum()}')

#### Diferença entre datas

In [ ]:
df_prazos = pd.DataFrame({
    'pedido':       ['P001', 'P002', 'P003'],
    'data_pedido':  ['2024-01-10', '2024-02-05', '2024-03-20'],
    'data_entrega': ['2024-01-18', '2024-02-12', '2024-04-02']
})

df_prazos['data_pedido']  = pd.to_datetime(df_prazos['data_pedido'])
df_prazos['data_entrega'] = pd.to_datetime(df_prazos['data_entrega'])
df_prazos['prazo_dias']   = (df_prazos['data_entrega'] - df_prazos['data_pedido']).dt.days

df_prazos

---
## Correlação de Pearson — `df.corr()` e `sns.heatmap()`

A correlação de Pearson mede a **força e a direção** da relação linear entre duas variáveis numéricas.

| Valor de r | Interpretação |
|---|---|
| ±1,0 | Correlação linear perfeita |
| \|r\| > 0,7 | Forte |
| 0,3 < \|r\| < 0,7 | Moderada |
| \|r\| < 0,3 | Fraca |
| 0 | Sem correlação linear |

> ⚠️ **Correlação ≠ causalidade.** Use apenas com variáveis numéricas contínuas e relação esperada linear.

In [ ]:
df = pd.DataFrame({
    'idade':      [22, 35, 41, 28, 55, 30, 47, 38, 26, 52],
    'salario':    [2800, 4500, 5200, 3100, 7000, 3800, 6100, 4900, 2600, 6800],
    'filhos':     [0, 1, 2, 0, 3, 1, 2, 1, 0, 3],
    'distancia':  [10, 25, 30, 8, 45, 15, 38, 22, 5, 42],
    'satisfacao': [8, 7, 6, 9, 5, 8, 6, 7, 9, 4]
})

df

#### Calcular a matriz de correlação

In [ ]:
corr = df.corr(method='pearson')
corr.round(2)

#### Correlação entre um par específico de variáveis

In [ ]:
r_idade_salario = df['idade'].corr(df['salario'])
r_dist_satisf   = df['distancia'].corr(df['satisfacao'])

print(f'r (idade × salário):        {r_idade_salario:.2f}')
print(f'r (distância × satisfação): {r_dist_satisf:.2f}')

#### Filtrar pares com correlação forte

In [ ]:
threshold = 0.6

pares = (
    corr.abs()
    .stack()
    .reset_index()
    .rename(columns={0: 'r', 'level_0': 'var1', 'level_1': 'var2'})
    .query('var1 < var2 and r >= @threshold')
    .sort_values('r', ascending=False)
)

print(f'Pares com |r| >= {threshold}:')
pares

#### Visualizar com `sns.heatmap()`

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    linecolor='white'
)
plt.title('Matriz de Correlação de Pearson')
plt.tight_layout()
plt.show()

#### Scatter plot — par com maior correlação

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(df['idade'], df['salario'], color='steelblue', edgecolors='white', s=80)
plt.xlabel('Idade')
plt.ylabel('Salário (R$)')
plt.title(f'Idade × Salário  |  r = {r_idade_salario:.2f}')
plt.tight_layout()
plt.show()

---
## Resumo Geral

| Operação | Método principal | Observação |
|---|---|---|
| Renomear colunas específicas | `df.rename(columns={...})` | Não altera as demais |
| Padronizar todos os nomes | `df.columns.str.lower()` | Ideal após leitura de CSV |
| Remover duplicatas | `df.drop_duplicates()` | Use `subset` e `keep` |
| Converter tipo | `df[col].astype(tipo)` | Cuidado com NaN |
| Tratar inválidos na conversão | `pd.to_numeric(errors='coerce')` | Gera NaN nos inválidos |
| Converter data | `pd.to_datetime(df[col])` | Use `format=` para datas BR |
| Extrair componentes de data | `df[col].dt.year`, `.dt.month`... | Requer dtype datetime |
| Matriz de correlação | `df.corr(method='pearson')` | Somente colunas numéricas |
| Visualizar correlação | `sns.heatmap(corr, annot=True)` | `cmap='coolwarm'` é padrão |